# Lab 1A: Introduction to DuckDB - SOLUTION

This notebook contains the solution for Lab 1A exercises.

## 💻 Exercise 1: DuckDB Use Case Analysis - SOLUTION

### Solution

In [ ]:
scenarios = [
    "Real-time inventory management for e-commerce",
    "Local data analysis for a data science project",
    "Multi-user business intelligence dashboard",
    "ETL pipeline for data warehouse",
    "Embedded analytics in a desktop application"
]

analysis = {
    "Real-time inventory management for e-commerce": {
        "fit": "Poor",
        "reason": "DuckDB is not optimized for high-concurrency transactional workloads. Use PostgreSQL or MySQL instead."
    },
    "Local data analysis for a data science project": {
        "fit": "Good",
        "reason": "DuckDB excels at local analytical workloads, integrates well with Python/pandas, and requires no server setup."
    },
    "Multi-user business intelligence dashboard": {
        "fit": "Marginal",
        "reason": "DuckDB has limited concurrent write support. Better for single-user or read-heavy scenarios. Consider cloud warehouses for multi-user BI."
    },
    "ETL pipeline for data warehouse": {
        "fit": "Good",
        "reason": "DuckDB is excellent for data transformation, ETL processing, and intermediate data storage in pipelines."
    },
    "Embedded analytics in a desktop application": {
        "fit": "Good",
        "reason": "DuckDB's embedded nature makes it perfect for desktop applications needing local analytics capabilities."
    }
}

for scenario in scenarios:
    result = analysis[scenario]
    print(f"Scenario: {scenario}")
    print(f"DuckDB fit: {result['fit']}")
    print(f"Reason: {result['reason']}")
    print()

## 💻 Exercise 2: Data Format Comparison - SOLUTION

### Solution

In [ ]:
import os
import duckdb
import time

con = duckdb.connect(':memory:')

# Create test data
con.execute("""
    CREATE TABLE test_data AS
    SELECT 
        (random() * 1000)::int as id,
        'Name ' || id as name,
        random() * 100 as value,
        '2023-01-01'::date + (random() * 365)::int as date
    FROM range(100000)
""")

# Test different formats
formats = ['csv', 'parquet', 'json']
results = []

for format in formats:
    # Export
    start = time.time()
    con.execute(f"COPY test_data TO 'data/test.{format}' (FORMAT {format.upper()})")
    export_time = time.time() - start
    
    # File size
    file_size = os.path.getsize(f'data/test.{format}') / 1024  # KB
    
    # Query performance
    start = time.time()
    con.execute(f"SELECT COUNT(*), AVG(value) FROM 'data/test.{format}'").fetchone()
    query_time = time.time() - start
    
    results.append({
        'format': format,
        'export_time': export_time,
        'file_size': file_size,
        'query_time': query_time
    })

# Display results
print("Data Format Comparison Results:")
print("="*50)
for result in results:
    print(f"\n{result['format'].upper()}:")
    print(f"  Export time: {result['export_time']:.4f}s")
    print(f"  File size: {result['file_size']:.2f}KB")
    print(f"  Query time: {result['query_time']:.4f}s")

# Analysis
print("\n" + "="*50)
print("Analysis:")
print("- Parquet typically offers the best compression and query performance")
print("- CSV is human-readable but larger and slower")
print("- JSON is flexible but less efficient for analytical workloads")
print("- For production analytics, Parquet is recommended")